In [8]:
import string
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import TfidfVectorizer

In [38]:
df = pd.read_csv("kitap.csv")
df = df.drop(["Okuma_sayisi","begenen_sayisi"], axis=1)
df = df.dropna()
df.head(5)

,Kitabin_Adi,Yazarin_Adi
0,Kürk Mantolu Madonna,Sabahattin_Ali
1,Küçük Prens,Antoine de Saint-Exupery
2,Satranç,Stefan Zweig
3,Dönüşüm,Franz Kafka
4,Şeker Portakalı,José Mauro de Vasconcelos


In [39]:
def trlower(text):
    result = ""
    for letter in text:
        if letter == "I":
            letter = "ı"
            result += letter
        elif letter == "İ":
            letter = "i"
            result += letter
        else:
            result += letter
    result = result.lower()
    return result

In [40]:
for col in df.columns:
    df[col] = df[col].apply(trlower)

In [41]:
def remove_punctuation(text):
    no_punc = [words for words in text if words not in string.punctuation]
    word_wo_punc = "".join(no_punc)
    return word_wo_punc

In [42]:
for col in df.columns:
    df[col] = df[col].apply(remove_punctuation)

In [43]:
df.head()

,Kitabin_Adi,Yazarin_Adi
0,kürk mantolu madonna,sabahattinali
1,küçük prens,antoine de saintexupery
2,satranç,stefan zweig
3,dönüşüm,franz kafka
4,şeker portakalı,josé mauro de vasconcelos


In [44]:
tfidf_matrix = TfidfVectorizer().fit_transform(df["Yazarin_Adi"])
tfidf_matrix

<56x102 sparse matrix of type '<class 'numpy.float64'>'
	with 114 stored elements in Compressed Sparse Row format>

In [45]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim

array([[1., 0., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 0., 1.]])

In [46]:
cosine_sim.shape

(56, 56)

In [47]:
indices = pd.Series(df.index, index=df["Kitabin_Adi"]).drop_duplicates()
indices

Kitabin_Adi
kürk mantolu madonna                 0
küçük prens                          1
satranç                              2
dönüşüm                              3
şeker portakalı                      4
simyacı                              5
hayvan çiftliği                      6
bilinmeyen bir kadının mektubu       7
uçurtma avcısı                       8
kuyucaklı yusuf                      9
insan neyle yaşar                   10
iki şehrin hikayesi                 11
yüzüklerin efendisi                 12
küçük prens                         13
hobbit                              14
kızıl köşkün rüyası                 15
on küçük zenci                      16
aslan cadı ve dolap                 17
ayişe                               18
da vinci şifresi                    19
gönülçelen                          20
simyacı                             21
lolita                              22
heidi                               23
bebek bakımı ve çocuk eğitimi       24
yeşilin kızı 

In [48]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key = lambda x : x[1], reverse = True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return df["Kitabin_Adi"].iloc[movie_indices]

In [49]:
df.iloc[30,:]

Kitabin_Adi    hite raporu
Yazarin_Adi     shere hite
Name: 30, dtype: object

In [50]:
df[29:30]

,Kitabin_Adi,Yazarin_Adi
29,watership tepesi,richard adams


In [51]:
gt = get_recommendations("xxxx")
df_gt = pd.DataFrame(columns = ["Kitabin_Adi","Yazarin_Adi"])

for index in gt.index:
    value = pd.DataFrame(df.iloc[index, 0:3]).T
    df_gt = pd.concat([df_gt, value], axis=0)

df_gt

,Kitabin_Adi,Yazarin_Adi
55,xxxx,sabahattinali
1,küçük prens,antoine de saintexupery
2,satranç,stefan zweig
3,dönüşüm,franz kafka
4,şeker portakalı,josé mauro de vasconcelos
5,simyacı,paulo coelho
6,hayvan çiftliği,george orwell
7,bilinmeyen bir kadının mektubu,stefan zweig
8,uçurtma avcısı,khaled hosseini
9,kuyucaklı yusuf,sabahattin ali
